# VII. Explore Realism Injection Behavior

This notebook evaluates how the injected realism changed the marketing dataset.

Goals:
- compare cleaned vs realistic data side by side
- inspect distribution, channel-level, and temporal shifts
- save useful charts and a final summary table to `../results/realism_exploration/`

## Step 1: Imports and Paths

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Define canonical input locations used across the project.
CLEANED_DATASET_PATH = Path("../datasets/cleaned_marketing_campaign_dataset.csv")
REALISTIC_DATASET_PATH = Path("../datasets/realistic_campaign_dataset.csv")

# Save all artifacts directly under the realism exploration results folder.
RESULTS_DIR = Path("../results/realism_exploration")
FINAL_SUMMARY_PATH = RESULTS_DIR / "final_summary.csv"

# Ensure the destination exists before any write operations.
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Cleaned dataset:   {CLEANED_DATASET_PATH.resolve()}")
print(f"Realistic dataset: {REALISTIC_DATASET_PATH.resolve()}")
print(f"Results folder:    {RESULTS_DIR.resolve()}")

Cleaned dataset:   C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\datasets\cleaned_marketing_campaign_dataset.csv
Realistic dataset: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\datasets\realistic_campaign_dataset.csv
Results folder:    C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\results\realism_exploration


## Step 2: Load and Validate Inputs

In [ ]:
# Fail fast if either required input is missing.
if not CLEANED_DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing cleaned dataset: {CLEANED_DATASET_PATH.resolve()}")

if not REALISTIC_DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing realistic dataset: {REALISTIC_DATASET_PATH.resolve()}")

# Parse Date immediately so temporal diagnostics can run without extra conversions.
clean_df = pd.read_csv(CLEANED_DATASET_PATH, parse_dates=["Date"])
real_df = pd.read_csv(REALISTIC_DATASET_PATH, parse_dates=["Date"])

required_columns = [
    "Date", "Channel_Used", "Campaign_Type", "Target_Audience",
    "Conversion_Rate", "Acquisition_Cost", "ROI",
    "Clicks", "Impressions", "Engagement_Score"
]

# Validate the minimum schema expected by downstream checks and charts.
missing_clean = [c for c in required_columns if c not in clean_df.columns]
missing_real = [c for c in required_columns if c not in real_df.columns]
if missing_clean:
    raise ValueError(f"Cleaned dataset missing columns: {missing_clean}")
if missing_real:
    raise ValueError(f"Realistic dataset missing columns: {missing_real}")

print(f"Cleaned shape:   {clean_df.shape}")
print(f"Realistic shape: {real_df.shape}")

pd.DataFrame({
    "dataset": ["cleaned", "realistic"],
    "rows": [len(clean_df), len(real_df)],
    "columns": [clean_df.shape[1], real_df.shape[1]],
    "duplicate_rows": [clean_df.duplicated().sum(), real_df.duplicated().sum()],
})

Cleaned shape:   (200000, 16)
Realistic shape: (200000, 16)


,dataset,rows,columns,duplicate_rows
0,cleaned,200000,16,0
1,realistic,200000,16,0


## Step 3: Numeric Shift Summary

This table gives a quick first-pass read of how the realism step changed central tendency and dispersion.

In [ ]:
# Core numeric metrics most likely to reveal realism effects.
numeric_cols = ["Conversion_Rate", "Acquisition_Cost", "ROI", "Clicks", "Impressions", "Engagement_Score"]

summary_rows = []
for col in numeric_cols:
    clean_mean = clean_df[col].mean()
    real_mean = real_df[col].mean()
    clean_std = clean_df[col].std()
    real_std = real_df[col].std()

    # Track both level and dispersion changes for each metric.
    summary_rows.append({
        "metric": col,
        "cleaned_mean": clean_mean,
        "realistic_mean": real_mean,
        "mean_pct_change": ((real_mean - clean_mean) / clean_mean * 100) if clean_mean != 0 else np.nan,
        "cleaned_std": clean_std,
        "realistic_std": real_std,
        "std_pct_change": ((real_std - clean_std) / clean_std * 100) if clean_std != 0 else np.nan,
    })

final_summary = pd.DataFrame(summary_rows).round(4)

# Persist summary for use in downstream reporting notebooks or scripts.
final_summary.to_csv(FINAL_SUMMARY_PATH, index=False)
print(f"Saved summary to: {FINAL_SUMMARY_PATH.resolve()}")

final_summary

Saved summary to: C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\results\realism_exploration\final_summary.csv


,metric,cleaned_mean,realistic_mean,mean_pct_change,cleaned_std,realistic_std,std_pct_change
0,Conversion_Rate,0.0801,0.0841,5.0922,0.0406,0.0474,16.8528
1,Acquisition_Cost,12504.3930,13404.5491,7.1987,4337.6645,5847.3062,34.8031
2,ROI,5.0024,5.2457,4.8627,1.7345,2.9753,71.5378
3,Clicks,549.7720,341.1483,-37.9473,260.0191,197.4833,-24.0505
4,Impressions,5507.3015,5569.7428,1.1338,2596.8643,3047.5102,17.3535
5,Engagement_Score,5.4947,5.4590,-0.6498,2.8726,2.9117,1.3610


## Step 4: Missingness and Relationship Checks

These checks are useful because realism injection intentionally introduced limited irregularities.

In [ ]:
# Compare missingness rates column-by-column between both datasets.
missing_compare = pd.DataFrame({
    "column": real_df.columns,
    "cleaned_missing_pct": clean_df.isna().mean().reindex(real_df.columns, fill_value=np.nan).values * 100,
    "realistic_missing_pct": real_df.isna().mean().values * 100,
}).round(3)

missing_compare["missing_pct_delta"] = (
    missing_compare["realistic_missing_pct"] - missing_compare["cleaned_missing_pct"]
).round(3)

# CTR is a practical sanity check for funnel consistency after injections.
ctr_clean = (clean_df["Clicks"] / clean_df["Impressions"].clip(lower=1)).clip(0, 1)
ctr_real = (real_df["Clicks"] / real_df["Impressions"].clip(lower=1)).clip(0, 1)

relationship_checks = pd.DataFrame([
    {
        "check": "Clicks-Impressions correlation",
        "cleaned": clean_df[["Clicks", "Impressions"]].corr().iloc[0, 1],
        "realistic": real_df[["Clicks", "Impressions"]].corr().iloc[0, 1],
    },
    {
        "check": "CTR median",
        "cleaned": ctr_clean.median(),
        "realistic": ctr_real.median(),
    },
    {
        "check": "ROI std",
        "cleaned": clean_df["ROI"].std(),
        "realistic": real_df["ROI"].std(),
    },
]).round(4)

print("Missingness comparison (top deltas):")
display(missing_compare.sort_values("missing_pct_delta", ascending=False).head(10))

print("Relationship checks:")
relationship_checks

Missingness comparison (top deltas):


,column,cleaned_missing_pct,realistic_missing_pct,missing_pct_delta
0,Campaign_ID,0.0,0.0,0.0
1,Company,0.0,0.0,0.0
2,Campaign_Type,0.0,0.0,0.0
3,Target_Audience,0.0,0.0,0.0
4,Duration,0.0,0.0,0.0
5,Channel_Used,0.0,0.0,0.0
6,Conversion_Rate,0.0,0.0,0.0
7,Acquisition_Cost,0.0,0.0,0.0
8,ROI,0.0,0.0,0.0
9,Location,0.0,0.0,0.0


Relationship checks:


,check,cleaned,realistic
0,Clicks-Impressions correlation,0.0000,-0.1777
1,CTR median,0.0998,0.0580
2,ROI std,1.7345,2.9753


## Step 5: Visual Diagnostics and Export

This cell saves three practical charts for reporting:
- ROI distribution comparison
- channel-level ROI spread
- click vs impression relationship

In [ ]:
plt.style.use("seaborn-v0_8-darkgrid")

# 1) ROI distribution comparison between baseline and realism-injected data.
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(clean_df["ROI"], bins=40, alpha=0.55, label="Cleaned")
ax.hist(real_df["ROI"], bins=40, alpha=0.55, label="Realistic")
ax.set_title("ROI Distribution: Cleaned vs Realistic")
ax.set_xlabel("ROI")
ax.set_ylabel("Frequency")
ax.legend()
roi_hist_path = RESULTS_DIR / "roi_distribution_cleaned_vs_realistic.png"
fig.tight_layout()
fig.savefig(roi_hist_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# 2) Channel-level ROI spread helps verify intended channel differentiation.
ordered_channels = real_df.groupby("Channel_Used")["ROI"].median().sort_values().index
plot_data = [real_df.loc[real_df["Channel_Used"] == ch, "ROI"].dropna().values for ch in ordered_channels]
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(plot_data, tick_labels=ordered_channels, patch_artist=True)
ax.set_title("Realistic ROI by Channel")
ax.set_xlabel("Channel Used")
ax.set_ylabel("ROI")
ax.tick_params(axis="x", rotation=30)
channel_boxplot_path = RESULTS_DIR / "realistic_roi_by_channel_boxplot.png"
fig.tight_layout()
fig.savefig(channel_boxplot_path, dpi=300, bbox_inches="tight")
plt.close(fig)

# 3) Sample scatter avoids overplotting while preserving structural signal.
sample_size = min(2500, len(real_df))
sample_df = real_df.sample(n=sample_size, random_state=42) if len(real_df) > sample_size else real_df
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sample_df["Impressions"], sample_df["Clicks"], alpha=0.35, s=20)
ax.set_title("Realistic Dataset: Clicks vs Impressions")
ax.set_xlabel("Impressions")
ax.set_ylabel("Clicks")
clicks_vs_impressions_path = RESULTS_DIR / "realistic_clicks_vs_impressions_scatter.png"
fig.tight_layout()
fig.savefig(clicks_vs_impressions_path, dpi=300, bbox_inches="tight")
plt.close(fig)

print("Charts saved to:")
print(f"- {roi_hist_path.resolve()}")
print(f"- {channel_boxplot_path.resolve()}")
print(f"- {clicks_vs_impressions_path.resolve()}")

C:\Users\hecto\AppData\Local\Temp\ipykernel_64220\1426197647.py:20: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot(plot_data, labels=ordered_channels, patch_artist=True)


Charts saved to:
- C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\results\realism_exploration\charts\roi_distribution_cleaned_vs_realistic.png
- C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\results\realism_exploration\charts\realistic_roi_by_channel_boxplot.png
- C:\Users\hecto\OneDrive\Documentos\VSCode\py_marketing_eda\results\realism_exploration\charts\realistic_clicks_vs_impressions_scatter.png


## Step 6: Interpretation Notes

Use this checklist when reviewing results:
- Did ROI variance increase in realistic data?
- Do channel medians show intentional efficiency differences?
- Is missingness mostly localized (for example, Engagement_Score)?
- Did click-impression correlation stay plausible while allowing more noise?